# Topic: SQL Date Difference (Date Arithmetic)

## Definition (30-second explanation)
* Date difference calculations measure the time elapsed between two specific timestamps or dates.
* They are heavily used in product and growth analytics to measure user lifecycles, conversion times, and subscription lengths.

## Why Interviewers Ask This
* Every database dialect (MySQL, PostgreSQL, SQL Server) handles date math differently. Interviewers want to see if you know your specific dialect's quirks.
* Tests your ability to handle `NULL` values (e.g., calculating lifetime for churned vs. active users).
* Evaluates your understanding of interval logic and real-world business metrics like "Time to First Action."

## Core Concepts
* **Dialect Differences:** PostgreSQL uses direct subtraction `(date2 - date1)` or `DATE_PART()`. MySQL uses `DATEDIFF(date2, date1)`.
* **Current Date:** Knowing how to call today's date (`CURRENT_DATE`, `CURDATE()`, or `GETDATE()`) is essential for "days since" calculations.
* **Intervals:** Using `INTERVAL` to add or subtract time periods (e.g., `date >= CURRENT_DATE - INTERVAL 30 DAY`).

## When to Use
* Calculating customer lifetime value (LTV) or days active.
* Finding the average gap between consecutive purchases or events.
* Filtering data to trailing windows (e.g., "users who signed up in the last 7 days").

## Advantages
* Date functions allow for dynamic queries that automatically update relative to `CURRENT_DATE` (e.g., rolling dashboards).
* Conditional logic (`CASE WHEN`) pairs perfectly with date math to handle multiple user states (active vs. churned).

## Limitations
* `DATEDIFF` logic strictly counts the boundaries crossed (days), which can sometimes ignore the time-of-day precision if DATETIME columns aren't truncated properly.
* Standard date math does not automatically exclude weekends or holidays (business days require calendar tables).

## Common Comparisons
* **DATETIME vs. DATE:** `DATETIME` includes hours/minutes/seconds. Comparing a `DATETIME` to a `DATE` can yield unexpected decimal results or off-by-one errors unless you use `DATE()` to truncate the timestamp first.

## Common Interview Traps
* **Argument Order Reversal:** MySQL uses `DATEDIFF(end, start)` but SQL Server uses `DATEDIFF(day, start, end)`. Getting this backward results in negative numbers.
* **Forgetting NULL Filters:** Doing math on `NULL` (e.g., a user who hasn't churned) returns `NULL`. You must use `WHERE date IS NOT NULL` or `COALESCE` / `CASE WHEN`.
* **Timezone Blindness:** Ignoring timezone differences when subtracting raw `TIMESTAMP` columns.

## Python / SQL Syntax (if applicable)
```sql
-- MySQL / SQLite approach
SELECT 
    user_id,
    CASE 
        WHEN churn_date IS NULL THEN DATEDIFF(CURDATE(), signup_date)
        ELSE DATEDIFF(churn_date, signup_date)
    END AS lifetime_days
FROM user_events;
```

## Important Formula (if applicable)
* **Relative Lookback:** `WHERE event_date >= CURRENT_DATE - INTERVAL '30' DAY`

## 45-Second Interview Answer
"For date difference calculations, the most important thing is knowing your specific SQL dialect, as PostgreSQL, MySQL, and SQL Server all have different syntax. I typically use `DATEDIFF()` or direct date subtraction to find the gap between two events. I always make sure to wrap my logic in a `CASE WHEN` to handle `NULL` values—like active users who don't have a churn date—and I use the `INTERVAL` keyword for dynamic trailing filters, like finding events in the last 30 days."

## Example Questions and Answers:

### Q1. Find the average days between order placement and delivery for each product category.

**Ideal Interview Answer (MySQL):**
```sql
SELECT 
    category,
    AVG(DATEDIFF(delivery_date, order_date)) AS avg_delivery_days
FROM orders
WHERE delivery_date IS NOT NULL
GROUP BY category;
```

**Common Mistakes Candidates Make:**
* Forgetting `WHERE delivery_date IS NOT NULL`. If an order is still in transit, `DATEDIFF` returns `NULL`, which might skew the average or cause errors depending on the database's strictness.
* Getting the `DATEDIFF` arguments backward, resulting in a negative average.

**One Likely Interviewer Follow-up:**
"How would you modify this to calculate the average *hours* instead of days, assuming these are DATETIME columns?"
*(Answer: In MySQL, use `TIMESTAMPDIFF(HOUR, order_date, delivery_date)`. In PostgreSQL, use `EXTRACT(EPOCH FROM (delivery_date - order_date))/3600`.)*

### Q2. Identify users who made their first purchase within 3 days of signing up.

**Ideal Interview Answer (MySQL):**
```sql
SELECT 
    user_id
FROM user_events
WHERE first_purchase_date IS NOT NULL 
  AND DATEDIFF(first_purchase_date, signup_date) <= 3;
```

**Common Mistakes Candidates Make:**
* Using `INTERVAL` incorrectly in the `WHERE` clause (e.g., `first_purchase_date <= signup_date + 3` works in some dialects but is safer using explicitly added intervals or DATEDIFF).
* Not excluding users who haven't purchased yet (`NULL` handling).

**One Likely Interviewer Follow-up:**
"If `first_purchase_date` is a full timestamp and `signup_date` is just a date, how do you ensure the 3-day window is accurate to the exact hour of signup?"
*(Answer: Ensure both columns are cast to exact timestamps, and calculate the difference in hours `(<= 72 hours)` rather than relying on calendar day boundaries.)*

### Q3. Calculate how many days each active subscription has been running.

**Ideal Interview Answer (MySQL):**
```sql
SELECT 
    subscription_id,
    DATEDIFF(CURDATE(), start_date) AS days_running
FROM subscriptions
WHERE status = 'active';
```

**Common Mistakes Candidates Make:**
* Hardcoding a specific date instead of using a dynamic function like `CURDATE()`, `CURRENT_DATE`, or `GETDATE()`.

**One Likely Interviewer Follow-up:**
"If this query is used for a daily reporting dashboard, what timezone does `CURDATE()` default to, and why might that be a problem?"
*(Answer: It defaults to the database server's timezone. If the server is in UTC but the business operates in PST, the 'current date' will roll over at 4 PM PST/5 PM PDT, causing inaccurate daily metrics. You should explicitly cast/convert to the business timezone.)*

### Q4. Find the average gap in days between consecutive purchases for each customer.

**Ideal Interview Answer (MySQL 8.0+):**
```sql
WITH PurchaseLags AS (
    SELECT 
        customer_id,
        purchase_date,
        LAG(purchase_date) OVER(PARTITION BY customer_id ORDER BY purchase_date) AS prev_purchase_date
    FROM purchases
)
SELECT 
    customer_id,
    AVG(DATEDIFF(purchase_date, prev_purchase_date)) AS avg_days_between_purchases
FROM PurchaseLags
WHERE prev_purchase_date IS NOT NULL
GROUP BY customer_id;
```

**Common Mistakes Candidates Make:**
* Trying to solve this with a self-join. While possible, self-joining to find "the immediate previous date" is highly inefficient and complex compared to the `LAG()` window function.
* Forgetting the `WHERE prev_purchase_date IS NOT NULL` in the outer query, which includes the first purchase's `NULL` lag in the data (though `AVG` usually ignores NULLs, it's safer to explicitly filter).

**One Likely Interviewer Follow-up:**
"What happens to customers who have only made a single purchase? Will they appear in this final output?"
*(Answer: No. For a single purchase, `LAG` returns NULL. The outer query filters out the NULL, leaving no records for that customer to `GROUP BY`, which is mathematically correct since they have no 'gap' to average.)*

## Practice Questions:

### Q1: Coding & Business Thinking (Sessionization)

**Scenario: You are analyzing user web traffic. A user can visit a website multiple times a day. The product team defines a "new session" as any page visit that occurs more than 30 minutes after the user's previous page visit.**

Data Setup (MySQL / SQLite):
```sql
CREATE TABLE web_traffic (
    user_id INT,
    visit_time DATETIME
);

INSERT INTO web_traffic (user_id, visit_time) VALUES
(1, '2026-07-31 10:00:00'),
(1, '2026-07-31 10:15:00'), -- Same session (15 mins later)
(1, '2026-07-31 11:00:00'), -- NEW session (45 mins later)
(2, '2026-07-31 09:00:00');
```

**Question: Write a query to flag whether each visit_time is the start of a new session (return 1 for a new session, 0 if it is part of the ongoing session). The very first visit for any user should always be flagged as 1. (You can use MySQL syntax like TIMESTAMPDIFF(MINUTE, start, end) or SQLite's (julianday(end) - julianday(start)) * 1440).**

**Answer (MySQL):**
```sql
WITH VisitLags AS (
    SELECT 
        user_id, 
        visit_time,
        LAG(visit_time) OVER (PARTITION BY user_id ORDER BY visit_time) AS prev_visit
    FROM web_traffic
)
SELECT 
    user_id, 
    visit_time,
    CASE 
        WHEN prev_visit IS NULL THEN 1
        WHEN TIMESTAMPDIFF(MINUTE, prev_visit, visit_time) > 30 THEN 1
        ELSE 0 
    END AS is_new_session
FROM VisitLags;
```

**Interview Tips:**
* **The "Sessionization" Pattern:** This is one of the most common advanced SQL questions at top tech companies. 
* **Next Step (Unique Session IDs):** Interviewers often ask you to take this one step further. If you wrap this entire query in another CTE, you can use `SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY visit_time)` to generate a unique, incrementing Session ID for every row!
* **First Event Handling:** Always explicitly handle `prev_visit IS NULL`. The very first event a user ever does is, by definition, the start of a session.